In [1]:

import os

# Use only one GPU to avoid DataParallel errors
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("GPU visibility configured!")


GPU visibility configured!


In [2]:

%pip install -q transformers datasets peft trl accelerate bitsandbytes


Note: you may need to restart the kernel to use updated packages.


In [3]:

import torch
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes

print("All libraries imported successfully!")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("Bitsandbytes:", bitsandbytes.__version__)

print("\nCUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )


All libraries imported successfully!
PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Datasets: 4.8.5
PEFT: 0.21.2
TRL: 1.18.2
Accelerate: 1.15.0
Bitsandbytes: 0.50.2

CUDA available: True
GPU count: 1
GPU name: Tesla T4
GPU memory: 14.56 GB


In [4]:

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Downloading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading 4-bit model...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("\nModel loaded successfully!")
print("Model:", model_name)
print("GPU:", torch.cuda.get_device_name(0))


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loading 4-bit model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
GPU: Tesla T4


In [5]:

from datasets import Dataset

qa_pairs = [
    ("What is TechZone?", "TechZone is a computer shop selling laptops, desktops, accessories, and computer components."),
    ("What products does TechZone sell?", "TechZone sells laptops, desktops, keyboards, mice, monitors, printers, and computer components."),
    ("Do you sell laptops?", "Yes, TechZone offers laptops for students, office workers, and gamers."),
    ("Do you sell desktop computers?", "Yes, we sell desktop computers for home, office, and gaming use."),
    ("Which laptop is suitable for students?", "A laptop with at least 8GB RAM, an SSD, and a modern processor is suitable for most student tasks."),
    ("Which laptop is best for gaming?", "A gaming laptop with a dedicated graphics card, at least 16GB RAM, and a powerful processor is recommended."),
    ("What is RAM?", "RAM is temporary memory that helps a computer run programs and multitask."),
    ("What is an SSD?", "An SSD is a storage device that provides faster data access than a traditional hard disk."),
    ("What is the difference between SSD and HDD?", "SSDs are generally faster, while HDDs usually offer more storage at a lower cost."),
    ("Do you sell keyboards?", "Yes, we sell wired and wireless keyboards."),
    ("Do you sell wireless mice?", "Yes, we offer wireless mice in different styles."),
    ("Do you sell gaming accessories?", "Yes, we sell gaming keyboards, mice, headsets, and other gaming accessories."),
    ("Do you sell monitors?", "Yes, we offer monitors in different sizes and resolutions."),
    ("Can I buy a printer?", "Yes, TechZone sells printers suitable for home and office use."),
    ("Do you provide computer repairs?", "Yes, TechZone provides basic computer troubleshooting and repair services."),
    ("My computer is running slowly. What should I do?", "Try restarting your computer, closing unnecessary applications, and checking available storage space."),
    ("My laptop is overheating. What should I do?", "Keep the air vents clear, use the laptop on a hard surface, and have the cooling system inspected if overheating continues."),
    ("My computer will not turn on. What should I do?", "Check the power cable, charger, battery, and power outlet. If it still does not turn on, seek technical assistance."),
    ("My laptop battery drains quickly. Why?", "High screen brightness, background applications, battery age, and power settings can affect battery life."),
    ("How can I protect my computer from viruses?", "Use reputable antivirus software, keep your operating system updated, and avoid suspicious downloads."),
    ("Do you provide software installation?", "Yes, we can assist with software installation and basic setup."),
    ("Can you upgrade my computer RAM?", "Yes, RAM upgrades are available depending on your computer's compatibility."),
    ("Can I upgrade my laptop storage?", "Many laptops support storage upgrades, but compatibility should be checked first."),
    ("What is the warranty period?", "For this sample dataset, assume eligible products include a one-year warranty. Actual warranty terms depend on the product."),
    ("Can I return a product?", "Returns are subject to the shop's return policy and the condition of the product."),
    ("Do you accept cash payments?", "Yes, the sample shop accepts cash payments."),
    ("Do you accept card payments?", "Yes, the sample shop accepts debit and credit card payments."),
    ("Can I reserve a laptop?", "You can contact TechZone to check availability and ask about reserving a laptop."),
    ("Do you offer delivery?", "Yes, delivery can be arranged depending on the customer's location."),
    ("Where can I get product information?", "You can contact TechZone's customer support team for product information."),
    ("What laptop specifications are good for office work?", "A modern processor, 8GB RAM, and an SSD are suitable for most office tasks."),
    ("What is a graphics card?", "A graphics card processes images and video and helps run graphics-intensive applications."),
    ("What is a motherboard?", "A motherboard is the main circuit board that connects the computer's components."),
    ("Why does my computer restart randomly?", "Possible causes include overheating, power problems, faulty memory, or software issues."),
    ("How often should I clean my computer?", "Clean external surfaces regularly and have internal dust cleaned periodically using safe procedures."),
    ("Can I connect a laptop to an external monitor?", "Yes, if the laptop has a compatible HDMI, DisplayPort, USB-C, or other video connection."),
    ("What is a gaming headset?", "A gaming headset combines headphones and a microphone for gaming communication and audio."),
    ("Do you sell USB flash drives?", "Yes, the sample shop offers USB flash drives in different storage capacities."),
    ("How can I make my laptop faster?", "You can reduce startup applications, free storage space, update software, and consider a RAM or SSD upgrade if supported."),
    ("How can I contact TechZone?", "You can contact the TechZone customer support team through the shop's available communication channels.")
]

data = []

for question, answer in qa_pairs:
    data.append({
        "messages": [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]
    })

dataset = Dataset.from_list(data)

print("Dataset created successfully!")
print("Total examples:", len(dataset))
print("\nFirst example:")
print(dataset[0])

dataset.to_json("computer_shop_dataset.json")

print("\nDataset saved successfully!")


Dataset created successfully!
Total examples: 40

First example:
{'messages': [{'content': 'What is TechZone?', 'role': 'user'}, {'content': 'TechZone is a computer shop selling laptops, desktops, accessories, and computer components.', 'role': 'assistant'}]}


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


Dataset saved successfully!


In [6]:

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("LoRA configuration completed!")


trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826
LoRA configuration completed!


In [7]:

from trl import SFTTrainer, SFTConfig

# Convert trainable LoRA parameters to FP32
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

# Training configuration
training_args = SFTConfig(
    output_dir="./computer_shop_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    max_length=512,
    max_grad_norm=0.3,
    packing=False,
    report_to="none"
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

print("Starting fine-tuning...")

train_result = trainer.train()

print("\nFine-tuning completed!")

# Save LoRA adapter
model.save_pretrained("./computer_shop_lora_final")
tokenizer.save_pretrained("./computer_shop_lora_final")

# Save training metrics
print("\nTraining metrics:")
print(train_result.metrics)

print("\nLoRA adapter saved successfully!")


Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Starting fine-tuning...


Step,Training Loss
5,2.938025
10,1.600395
15,1.142023



Fine-tuning completed!

Training metrics:
{'train_runtime': 37.4874, 'train_samples_per_second': 3.201, 'train_steps_per_second': 0.4, 'total_flos': 14751293446656.0, 'train_loss': 1.893480936686198, 'epoch': 3.0}

LoRA adapter saved successfully!


In [8]:

import torch

model.eval()

test_questions = [
    "What products does TechZone sell?",
    "Which laptop would you recommend for a university student studying programming?",
    "My laptop is overheating. What should I do?",
    "What should I consider before buying a laptop for video editing?"
]

def generate_answer(question):

    messages = [
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[-1]

    answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return answer


print("========== MODEL COMPARISON ==========")

comparison_results = []

for question in test_questions:

    print("\nQUESTION:", question)

    # Test original model without LoRA
    with model.disable_adapter():
        original_answer = generate_answer(question)

    # Test fine-tuned model with LoRA
    finetuned_answer = generate_answer(question)

    print("\nORIGINAL MODEL:")
    print(original_answer)

    print("\nFINE-TUNED MODEL:")
    print(finetuned_answer)

    comparison_results.append({
        "question": question,
        "original": original_answer,
        "fine_tuned": finetuned_answer
    })

    print("\n" + "=" * 60)

print("\nAll comparison tests completed!")


========== MODEL COMPARISON ==========

QUESTION: What products does TechZone sell?

ORIGINAL MODEL:
I'm sorry for the confusion, but I am not able to provide information on specific companies or products as I am a text-based AI model and do not have access to real-time data. However, I can suggest you check out the official website of TechZone or other reliable sources for more detailed information about their offerings.

FINE-TUNED MODEL:
TechZone sells various products such as servers, storage devices, networking equipment, and software solutions.


QUESTION: Which laptop would you recommend for a university student studying programming?

ORIGINAL MODEL:
As an AI language model, I don't have personal preferences or biases, but I can provide some general recommendations based on common standards and user feedback.

For a university student studying programming, it's important to consider factors such as the hardware specifications required (CPU, memory, storage), software requirement

In [9]:

import json

# Save comparison results
with open("model_comparison_results.json", "w") as f:
    json.dump(comparison_results, f, indent=4)

# Save final adapter again
model.save_pretrained("./computer_shop_lora_final")
tokenizer.save_pretrained("./computer_shop_lora_final")

print("All final results saved successfully!")
print("Files:")
print("1. computer_shop_dataset.json")
print("2. model_comparison_results.json")
print("3. computer_shop_lora_final/")


All final results saved successfully!
Files:
1. computer_shop_dataset.json
2. model_comparison_results.json
3. computer_shop_lora_final/
